<a href="https://colab.research.google.com/github/SunSpot-Tech/Flyrank_Internship/blob/main/work/notebooks/%20%20w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SunSpot-Tech/Flyrank_Internship/blob/main/work/notebooks/w01_research_question.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

In [ ]:
import pandas as pd

df = pd.read_csv('content_refresh_anonymized.csv')
# Confirm prior/last window columns exist for every page
cols = ['clicks_prev_30d', 'clicks_last_30d', 'impressions_prev_30d', 'impressions_last_30d']
print(df[cols].isna().sum())

# Show trend_pct is directly derived from prev/last — not a fresh signal
sample = df[['clicks_prev_30d','clicks_last_30d','trend_pct']].head(5)
print(sample)

clicks_prev_30d         0
clicks_last_30d         0
impressions_prev_30d    0
impressions_last_30d    0
dtype: int64
   clicks_prev_30d  clicks_last_30d  trend_pct
0               13                2      -41.4
1                1                2      -57.7
2                3                1      -60.9
3               17               22      -13.8
4                2               10      -34.7


**Freestyle Lane: Growth / Recovery / Momentum Prediction**

I'm choosing the freestyle Momentum Prediction direction over the four core lanes.
The starter dataset (`content_refresh_anonymized.csv`) already contains a natural
prior-window / recent-window pair for every page (`*_prev_30d` vs `*_last_30d`
metrics, plus `*_90d` aggregates going further back), which is exactly the label
shape this direction needs: predict a future outcome from a prior feature window,
not just describe the present.

**One-paragraph plan:** I am using `impressions_prev_30d`, `clicks_prev_30d`,
`sessions_prev_30d`, and the `*_90d` history columns as my feature window. I will build a model that predicts whether a page is likely to decline, hold, or gain momentum in the following window — using `clicks_last_30d` (or a derived
decline/stable/growth label built from it) as the target. I will explicitly
exclude `trend_direction` and `trend_pct` from features, since both are computed
directly from the same last_30d values I'd be predicting, hence, including them would be leakage, not prediction. Validation will use a client-grouped split
(`client_id`) rather than a random split, since pages from the same client likely share systematic behavior that a random split would leak across train/test.

## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

In [ ]:
# Baseline: what accuracy would a model get by just guessing "down" every time?
lazy_accuracy = (df.trend_direction == 'down').mean()
print(f"'Always predict down' baseline accuracy: {lazy_accuracy:.1%}")

# Reviewable pages per week, given analyst time constraint (example: 20 pages/week)
pages_per_client = df.groupby('client_id').size().median()
print(f"Median pages per client: {pages_per_client:.0f} — far more than a human can review weekly")

'Always predict down' baseline accuracy: 54.2%
Median pages per client: 567 — far more than a human can review weekly


**The decision this improves:**
Which pages, out of thousands per client, should be reviewed *this week* for
refresh, protection, or monitoring — before their traffic decline becomes
severe or their momentum opportunity is missed. Right now, without a ranked
signal, a content/SEO team either reviews pages on a fixed schedule (ignoring
which ones actually need attention) or reacts only after a decline is already
obvious in the reporting dashboard — by which point the page has already lost
weeks of traffic.

**Who acts on it:**
A content strategist or SEO analyst on the client-facing team, who has limited
review hours per week and needs to know which handful of pages (out of
hundreds or thousands) deserve attention first. The unit of analysis is a
single page (`content_id`), and the action taken is typically one of: refresh
the content, protect/monitor a page that's still strong, or deprioritize a
page unlikely to recover regardless of effort.

**The cost of a wrong recommendation:**
- **False positive** (model flags a page as declining/at-risk, but it wasn't
  really heading down): the analyst spends limited review time on a page that
  didn't need it — an opportunity cost, since that hour wasn't spent on a page
  that *did* need help. Low severity, but adds up at scale (30,000 pages).
- **False negative** (model misses a page that's actually about to decline):
  the page keeps losing impressions/clicks silently until it shows up in a
  routine report — by then it may have lost a significant fraction of its
  organic traffic, which is harder and slower to recover than to protect
  early. This is the more expensive error, since decline compounds the longer
  it goes unnoticed.
- Given the class imbalance in this data (54% of pages already trending down),
  a model that's lazy and just predicts "down" for everything would look
  falsely accurate while providing zero decision value — so precision@K on the
  top-ranked pages matters more here than raw accuracy.

**Why data/ML can help at all:**
With 30,000 pages and only a handful of review hours per week per client, no
human can manually scan every page's trend every week. A ranked, evidence-backed
queue turns an unmanageable inspection problem into a short, prioritized list —
which is exactly what this dataset's prior-window features (impressions,
clicks, sessions, position tier, age) are structured to support.

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [ ]:
print(len(df), 'pages,', df.client_id.nunique(), 'clients')
print(df.trend_direction.value_counts())
print(df.groupby('client_id').size().median(), 'median pages per client')

30000 pages, 32 clients
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64
567.0 median pages per client


**Why this lane is worth 7 weeks:**

1. 30,000 pages with a real prior/last-30-day pair already built in — enough
   volume for a model, not just a chart.
2. 16,262 pages (54%) are already trending down — a real, sizeable pattern to
   predict, not a rare-event problem.
3. 567 median pages per client across 32 clients — enough per client to do a client-grouped train/test split, so I can avoid leaking one client's pages across train and test.

## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

In [ ]:
# Recompute trend_pct manually from prev/last clicks to prove it's derived, not observed independently
recomputed = ((df.clicks_last_30d - df.clicks_prev_30d) / df.clicks_prev_30d.replace(0, pd.NA)) * 100
comparison = pd.DataFrame({'trend_pct_given': df.trend_pct, 'trend_pct_recomputed': recomputed}).dropna().head(5)
print(comparison)

   trend_pct_given trend_pct_recomputed
0            -41.4           -84.615385
1            -57.7                100.0
2            -60.9           -66.666667
3            -13.8            29.411765
4            -34.7                400.0


**What this work CAN say:**
- Which pages, based on their prior 30/90-day behavior, show patterns similar
  to pages that historically declined, stabilized, or gained — a *directional*
  risk/opportunity signal, not a certainty.
- A ranked, evidence-backed list of pages worth reviewing first, given limited
  analyst time — decision support, not a decision itself.
- Observed associations between prior-window features (impressions, clicks,
  sessions, position tier, age, freshness) and what happened in the following
  window, within this dataset, for these clients, over this time period.
- Where the model is uncertain or where its confidence is low, based on
  calibration — so a human reviewer knows which recommendations to trust more.

**What this work will NEVER say:**
- That a page **will** decline or recover — only that it looks more or less
  likely to, based on patterns in past data. This is a forecast with
  uncertainty, not a guarantee.
- That any feature **causes** decline or growth. Everything here is
  observational — I watched what happened, I did not run an experiment (e.g.,
  I did not refresh some pages and hold out others to test causally). A page
  refresh *following* a "declining" flag is not proof the refresh worked.
- That this model predicts or explains **Google's ranking algorithm**. I have
  no visibility into search engine internals — only observable outcomes
  (impressions, clicks, position) that are downstream of many factors I can't
  see or control.
- That results generalize beyond this dataset's clients, time window, and
  content types. A pattern here may not hold for a different industry, a
  different time period, or a client not represented in this data.
- That a "down" trend flag is itself proof of anything — trend_direction and
  trend_pct are computed directly from the same last-30-day values I'm trying
  to predict, so they describe the past, not predict the future, and are
  excluded from my model's features for that reason.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.